<a href="https://colab.research.google.com/github/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/03_gradio_rakam_tanima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a> &nbsp; <a href="https://github.com/DrMuratAltun/meb-yegitek-derin-ogrenme/blob/main/03_gradio_rakam_tanima.ipynb" target="_parent"><img src="https://img.shields.io/badge/GitHub-Kaynak-181717?logo=github" alt="GitHub'da Görüntüle"/></a>

> 🚀 **Bu notebook'u tarayıcıda çalıştırmak için sol üstteki "Colab'da Aç" butonuna tıklayın** — hiçbir şey kurmanıza gerek yok!


<div style="background: linear-gradient(135deg, #0B3D91 0%, #1565C0 60%, #1E88E5 100%); padding: 26px 30px; border-radius: 14px; color: white; margin: 0 0 24px 0; font-family: Georgia, serif; box-shadow: 0 4px 14px rgba(11,61,145,0.25);">

<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
  <span style="font-size: 11px; font-weight: bold; letter-spacing: 2px; color: #BBDEFB;">MEB · YEĞİTEK · DERİN ÖĞRENME SERİSİ</span>
  <span style="font-size: 11px; background: rgba(255,255,255,0.18); padding: 5px 12px; border-radius: 20px; color: white; font-weight: bold;">DERS 3 / 5</span>
</div>

<h1 style="margin: 8px 0 4px 0; color: white; font-size: 28px; line-height: 1.2;">✍️ Gradio ile Rakam Tanıma Uygulaması</h1>

<p style="margin: 8px 0 0 0; color: #E3F2FD; font-size: 13px;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong> — Öğretmenler için Uygulamalı Yapay Zeka<br>
<span style="color: #BBDEFB;">⏱️ Süre: ~50 dakika · 🖥️ Ortam: Google Colab veya yerel Python</span>
</p>

<div style="border-top: 1px solid rgba(255,255,255,0.22); margin-top: 14px; padding-top: 12px; font-size: 12px; color: #E3F2FD;">
🎯 Bu notebook <strong>non-teknik kitle</strong> için hazırlanmıştır · Kavramlar günlük hayattan analojilerle anlatılır · Kod minimum, açıklama maksimumdur.
</div>

</div>

## 🎯 Bu Notebook'ta Ne Öğreneceğiz?

Şimdiye kadar modeller eğittik, ama hep notebook içinde kaldı. Bu notebook'ta modelinizi **gerçek bir uygulamaya** dönüştüreceğiz!

Sonunda şunları yapabiliyor olacaksınız:

- 🖼️ Görüntü verisi (MNIST) ile sinir ağı eğitmek
- 💾 **Eğitilmiş modeli dosyaya kaydetmek** (`.keras` dosyası)
- 📂 **Kaydedilmiş modeli geri yüklemek**
- 🌐 **Gradio** ile bir web arayüzü açmak
- 🎨 Tarayıcıda **rakam çiz → tahmin gör** uygulaması yapmak

Bu, gerçek dünyada AI ürünlerinin nasıl çalıştığının tam bir minyatürü!

## 🤔 Önce Bir Senaryo

Postacı düşünün. Türkiye'de PTT, posta kodlarını **el yazısı** olarak okumak zorunda. Tabii artık otomatik makineler yapıyor — ama bu makinelerin arkasında ne var?

Cevap: **Sinir ağları**. Onlara binlerce el yazısı rakam gösterilir, sonra her zarfı saniyeler içinde tanırlar.

Biz de aynısını yapacağız!

---

### 📚 MNIST Veri Seti

MNIST, derin öğrenmenin "Merhaba Dünya"sıdır. İçinde:
- **70.000** adet 28×28 piksel **el yazısı rakam** (0-9) görüntüsü var
- **60.000'i** eğitim, **10.000'i** test için ayrılmış
- Görüntüler **siyah-beyaz** (gri tonlamalı)

Bu veri seti Keras'ta hazır geliyor — tek satır yükleyeceğiz.


## 📦 Kurulum

> ⚠️ **Önemli:** Gradio Colab'da otomatik yüklü değil. Aşağıdaki ilk hücre Gradio'yu yükler — sonra kütüphaneleri import ederiz.

In [ ]:
# Gradio'yu yükle (Colab'da gerekir)
!pip install -q gradio

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
import gradio as gr

keras.utils.set_random_seed(42)
print("✅ Hazır! Gradio sürümü:", gr.__version__)

## 📊 Veriyi Tanıyalım

In [ ]:
# MNIST'i yükle (otomatik internetten indirilir, sonra cache'lenir)
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"📦 Eğitim verisi: {X_train.shape}  →  {X_train.shape[0]} resim, her biri {X_train.shape[1]}x{X_train.shape[2]} piksel")
print(f"📦 Test verisi  : {X_test.shape}")
print(f"🏷️  Etiketler  : 0-9 arası rakamlar")

### Birkaç Örnek Görelim

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Etiket: {y_train[i]}', fontsize=11)
    ax.axis('off')
plt.suptitle('🔢 MNIST — İlk 10 Eğitim Örneği', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Veri Önişleme (Çok Önemli!)

Sinir ağları **0-1 arasındaki** sayıları sever. Şu an pikseller 0-255 arasında.

Yapacağımız:
1. **Normalize et:** 255'e böl, böylece her piksel 0.0-1.0 arasına gelir.
2. **Bir boyut ekle:** Keras "kaç görüntü, yükseklik, genişlik, kanal" formatı bekler.

> 🎓 **Analoji:** Tariflerde bazen "1 bardak un" yerine "100 gram un" yazar. Aynı bilgi, farklı ölçek. Sinir ağına da en sevdiği ölçekle vermek lazım.


In [ ]:
# 0-255 → 0-1 arasına ölçekle
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

print(f"✅ Normalize edildi. Min: {X_train.min()}, Max: {X_train.max()}")

## 🧠 Modeli Kuralım

Bu kez **görüntü** ile çalışıyoruz ama henüz CNN değil — düz Dense katmanlar yeterli (sıradaki notebook'ta CNN'e geçeceğiz).

Plan:
1. **Flatten:** 28×28 görüntüyü 784 elemanlı düz bir vektöre çevir.
2. **Dense (128, ReLU):** 128 nöronlu gizli katman.
3. **Dropout (0.2):** Aşırı öğrenmeyi (overfitting) önlemek için %20 nöronu rastgele kapat.
4. **Dense (10, Softmax):** 10 rakam için olasılık çıktısı.

> 🎓 **Dropout ne işe yarar?** Modelin tek bir özelliğe aşırı bağlanmasını engeller. Sınıfta her gün aynı öğrenciye soru sorarsanız sadece o öğrenci öğrenir. Rastgele soru sorarsanız herkes öğrenir.


In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),                                 # 28x28 → 784 vektör
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(10, activation='softmax')            # 10 sınıf (0-9)
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 🚀 Eğitim

5 epoch yeterli — MNIST göreli kolay bir veri seti.

In [ ]:
gecmis = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Eğitim grafiği
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(gecmis.history['accuracy'], label='Eğitim', linewidth=2)
axes[0].plot(gecmis.history['val_accuracy'], label='Doğrulama', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Doğruluk')
axes[0].set_title('Doğruluk', fontweight='bold'); axes[0].legend()

axes[1].plot(gecmis.history['loss'], label='Eğitim', linewidth=2)
axes[1].plot(gecmis.history['val_loss'], label='Doğrulama', linewidth=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Hata')
axes[1].set_title('Hata', fontweight='bold'); axes[1].legend()

plt.tight_layout()
plt.show()

## 🔍 Test Doğruluğu

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"🎯 Test doğruluğu: {test_acc * 100:.2f}%")
print(f"   {len(X_test)} test örneğinden ~{int(test_acc * len(X_test))} tanesi doğru!")

### Birkaç Tahmin Görelim

In [ ]:
tahminler = model.predict(X_test[:10], verbose=0)

fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    tahmin = np.argmax(tahminler[i])
    gercek = y_test[i]
    renk = 'green' if tahmin == gercek else 'red'
    ax.imshow(X_test[i], cmap='gray')
    ax.set_title(f'Tahmin: {tahmin} | Gerçek: {gercek}', fontsize=10, color=renk)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 💾 Modeli Kaydedelim

Bu, en önemli adım! Bütün eğitimi yaptık — **kaybetmek istemeyiz**.

Keras modelini tek satırla `.keras` dosyasına kaydediyoruz:

In [ ]:
# Modeli kaydet
model.save('rakam_modeli.keras')

import os
boyut = os.path.getsize('rakam_modeli.keras') / 1024
print(f"✅ Model kaydedildi: rakam_modeli.keras ({boyut:.1f} KB)")
print()
print("Bu dosya artık ağırlıkları + mimariyi içeriyor.")
print("Bilgisayarınızı kapatıp açsanız bile bu modeli yükleyip kullanabilirsiniz.")

## 📂 Modeli Geri Yükleyelim

Şimdi bilgisayarınızı yeni açtınız diyelim. Yapılacak tek şey:

In [ ]:
# Yeni bir değişkene yükle
yuklenen_model = keras.models.load_model('rakam_modeli.keras')

# Aynı tahminleri veriyor mu kontrol et
yeni_tahmin = yuklenen_model.predict(X_test[:1], verbose=0)
print(f"İlk test örneğinin tahmini: {np.argmax(yeni_tahmin)}")
print(f"Gerçek değer: {y_test[0]}")
print()
print("✅ Yüklenen model çalışıyor — tıpkı eğittiğimiz gibi!")

## 🌐 Gradio ile Web Arayüzü

Şimdi en eğlenceli bölüm: **Tarayıcıda rakam çiz → tahmin gör!**

### Gradio Nedir?

Gradio, makine öğrenmesi modellerine **5 dakikada web arayüzü** ekleyen bir kütüphane. Aşağıdaki adımları yapacağız:

1. Tahmin yapan bir Python fonksiyonu yaz
2. Bu fonksiyonu Gradio'ya ver
3. Gradio otomatik olarak web sayfası açsın

> 🎓 **Tarif:** Sen "girişte resim al, çıkışta yazı ver" dersin, Gradio kalanı halleder.


In [ ]:
def rakami_tahmin_et(cizim):
    """Gradio Sketchpad'in verdiği veriyi alıp tahmin yapar."""
    # Boş çizim kontrolü
    if cizim is None:
        return {str(i): 0.0 for i in range(10)}

    # Sketchpad bir dict döner: {'composite': ..., 'background': ..., 'layers': [...]}
    if isinstance(cizim, dict):
        img = cizim.get('composite', cizim.get('image'))
    else:
        img = cizim

    if img is None:
        return {str(i): 0.0 for i in range(10)}

    img = np.array(img)

    # RGBA ise gri tonlamaya çevir
    if img.ndim == 3:
        # Alpha kanalını kullan (çizilen yer = 255, boş = 0)
        if img.shape[-1] == 4:
            img = img[..., 3]
        else:
            img = img.mean(axis=-1)

    # 28x28 boyutuna küçült
    from PIL import Image as PILImage
    pil = PILImage.fromarray(img.astype('uint8'))
    pil = pil.resize((28, 28))
    img28 = np.array(pil).astype('float32') / 255.0

    # Tahmin
    olasiliklar = yuklenen_model.predict(img28.reshape(1, 28, 28), verbose=0)[0]
    return {str(i): float(olasiliklar[i]) for i in range(10)}


# Gradio arayüzü
arayuz = gr.Interface(
    fn=rakami_tahmin_et,
    inputs=gr.Sketchpad(label="✍️ Buraya bir rakam çizin (0-9)",
                        image_mode='L', type='numpy',
                        height=350, width=350,
                        brush=gr.Brush(default_size=20, colors=['#FFFFFF'], color_mode='fixed')),
    outputs=gr.Label(num_top_classes=3, label='🎯 En Olası 3 Tahmin'),
    title='✍️ MEB YEĞİTEK · El Yazısı Rakam Tanıma',
    description='Aşağıya **siyah arka plan üzerine BEYAZ** ile bir rakam (0-9) çizin. Eğitilmiş model tahmin edecek.',
    flagging_mode='never'
)

print("🌐 Gradio arayüzü hazırlandı. Şimdi launch() ile başlatacağız...")

### Arayüzü Başlatalım!

`launch()` çağrısı bir web sunucusu açar:
- **Yerel makinada:** http://127.0.0.1:7860
- **Colab'da:** Otomatik tıklanabilir bağlantı verir

> 💡 Çizim yaparken **MNIST'te rakamlar beyaz, arka plan siyah** olduğu için Sketchpad'i de öyle ayarladık. Eğer tahminler yanlış geliyorsa bu sebeple — gerçek bir uygulamada görüntüyü daha akıllı işlemek gerekir.

In [ ]:
arayuz.launch(share=False)   # share=True yaparsanız Gradio internette geçici link verir

## 📝 Özet

Bu notebook'ta **gerçek bir uygulama** yaptık:

| Adım | Komut |
|---|---|
| Modeli eğit | `model.fit(...)` |
| Modeli kaydet | `model.save('dosya.keras')` |
| Modeli yükle | `keras.models.load_model('dosya.keras')` |
| Web arayüzü oluştur | `gr.Interface(fn=..., inputs=..., outputs=...)` |
| Arayüzü başlat | `arayuz.launch()` |

### 🎓 Anahtar İçgörü

> **AI ürünü yapmak = Model + Arayüz + Persistans (kaydetme).**

Buraya kadar görüntülerle çalıştık ama **düz Dense katmanlarla**. Sıradaki notebook'ta görüntülere özel **Konvolüsyonel Sinir Ağları (CNN)** ile çok daha iyi sonuçlar alacağız! 🎨

<div style="background: #0B3D91; color: #BBDEFB; padding: 26px 30px; border-radius: 14px; margin-top: 32px; font-family: Georgia, serif;">

<h3 style="color: white; margin: 0 0 12px 0; border-bottom: 2px solid #FFC107; padding-bottom: 8px; display: inline-block;">MEB · YEĞİTEK</h3>

<p style="font-family: Calibri, sans-serif; font-size: 14px; color: #E3F2FD; margin: 0 0 8px 0; line-height: 1.6;">
<strong>Yenilik ve Eğitim Teknolojileri Genel Müdürlüğü</strong><br>
Öğretmen ve eğitimciler için uygulamalı derin öğrenme eğitim serisi.
</p>

<p style="font-family: Calibri, sans-serif; font-size: 12px; color: #BBDEFB; margin: 12px 0 0 0;">
🎓 Bu notebook eğitim amaçlıdır · Kullanılan tüm kütüphaneler açık kaynaktır.
</p>

<div style="background: rgba(255,255,255,0.08); border-left: 3px solid #FFC107; padding: 12px 16px; margin-top: 16px; font-family: Calibri, sans-serif; font-size: 13px; color: #FFF9E6;">
➡️ <strong>Bir sonraki notebook:</strong> <code>04_cnn_giysi_siniflandirma.ipynb</code> — Görüntü için özel tasarlanmış CNN'lerle giysi sınıflandırma.
</div>
<p style="font-family: Calibri, sans-serif; font-size: 11px; color: #90CAF9; margin: 14px 0 0 0; text-align: center; border-top: 1px solid rgba(255,255,255,0.15); padding-top: 12px;">
© 2026 MEB YEĞİTEK · Derin Öğrenme Serisi
</p>

</div>